# Furniture_Keras_Adam_Dropout

이 노트북은 발표용 재실험을 위해 **한 가지 조건만 명확하게 바꾸고**,  
학습 과정과 결과를 동일한 형식으로 저장하도록 구성했습니다.

## CNN 흐름과 코드 위치
- **[1] 데이터 입력**
- **[2] 특징 추출 - Conv Block 1**
- **[3] 특징 추출 - Conv Block 2**
- **[4] 추가 특징 추출 / 분류**
- **[5] Optimizer 설정**
- **[6] 학습**
- **[7] 성능 평가**
- **[8] 결과 저장**

> Keras와 PyTorch 모두 같은 번호 체계를 사용해 PPT에서 코드 위치를 비교하기 쉽게 만들었습니다.

## 이 실험에서 바뀌는 것

- Dataset: **Furniture**
- Framework: **Keras**
- Optimizer: **Adam**
- Version: **Dropout**
- Epoch: **30**
- Dropout: **0.5**
- Spatial Dropout: **사용하지 않음**
- FC Units: **128**

각 단계는 이전 단계에 변경사항을 **하나씩 누적**합니다.

`Baseline → Dropout → SpatialDropout → FC64`

In [ ]:
# 공통 설정
import os, time, random, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# [0] Google Drive 연결 + furniture.zip 압축 해제
#
# 내 드라이브 기준 위치:
# MyDrive/dataset/furniture.zip
#
# 압축 내부가
#   train / val / test
# 이거나
#   furniture/train / furniture/val / furniture/test
# 처럼 한 단계 더 들어 있어도 자동으로 찾아냅니다.
# ------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = Path("/content/drive/MyDrive/dataset/furniture.zip")
EXTRACT_DIR = Path("/content/furniture_dataset")

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"데이터셋 ZIP을 찾을 수 없습니다: {ZIP_PATH}\n"
        "Google Drive의 MyDrive/dataset/furniture.zip 위치를 확인하세요."
    )

# 이전 실행에서 풀어둔 파일이 있으면 지우고 다시 압축 해제
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

shutil.unpack_archive(str(ZIP_PATH), str(EXTRACT_DIR))
print("압축 해제 완료:", EXTRACT_DIR)

# train / val / test 폴더가 모두 들어 있는 실제 데이터 루트 자동 탐색
candidate_roots = [EXTRACT_DIR] + [
    p for p in EXTRACT_DIR.rglob("*") if p.is_dir()
]

valid_roots = [
    p for p in candidate_roots
    if all((p / split).is_dir() for split in ("train", "valid", "test"))
]

if not valid_roots:
    print("\n압축 해제 후 확인된 상위 폴더:")
    for p in sorted(EXTRACT_DIR.iterdir()):
        print(" -", p)
    raise FileNotFoundError(
        "압축 내부에서 train / val / test 폴더를 모두 포함한 위치를 찾지 못했습니다."
    )

# 가장 바깥쪽의 유효한 루트를 사용
DATASET_ROOT = min(valid_roots, key=lambda p: len(p.parts))

print("Dataset ZIP :", ZIP_PATH)
print("Dataset root:", DATASET_ROOT)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

SEED = 42
EPOCHS = 30
BATCH_SIZE = 64
IMAGE_SIZE = (128, 128)
NUM_CLASSES = 5
LEARNING_RATE = 0.001
MOMENTUM = 0.9
EXPERIMENT_NAME = "Furniture_Keras_Adam_Dropout"

OUTPUT_DIR = Path("/content/experiment_outputs") / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

print("Experiment:", EXPERIMENT_NAME)
print("Dataset root:", DATASET_ROOT)
print("Output:", OUTPUT_DIR)

## [1] 데이터 입력

가구 데이터셋은 `train / val / test`가 이미 분리되어 있다고 가정합니다.  
클래스 순서는 발표 자료와 동일하게 `almirah / chair / fridge / table / tv`로 고정합니다.

In [ ]:
# [1] 데이터 입력
CLASS_NAMES = ['almirah', 'chair', 'fridge', 'table', 'tv']

for split in ["train", "valid", "test"]:
    split_path = DATASET_ROOT / split
    if not split_path.exists():
        raise FileNotFoundError(
            f"{split_path} 폴더가 없습니다. DATASET_ROOT를 실제 Colab 경로로 수정하세요."
        )

train_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT / "train",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT / "valid",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = keras.utils.image_dataset_from_directory(
    DATASET_ROOT / "test",
    labels="inferred",
    label_mode="int",
    class_names=CLASS_NAMES,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Classes:", CLASS_NAMES)

## [2]~[4] CNN 모델 정의

Furniture는 Fashion-MNIST보다 입력이 크고 RGB이므로 Conv Block을 3개 사용합니다.

`Input → Conv32 → BN → ReLU → Pool → Conv64 → BN → ReLU → Pool → Conv128 → BN → ReLU → Pool → GAP → FC → Dropout → Output`

In [ ]:
model = keras.Sequential([
    # [1] 입력: 128×128 RGB 이미지
    layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)),
    layers.Rescaling(1.0 / 255.0),

    # [2] 특징 추출 - Conv Block 1
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(2),


    # [3] 특징 추출 - Conv Block 2
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(2),


    # [4] 추가 특징 추출 - Conv Block 3
    layers.Conv2D(128, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(2),

    # [4] 분류
    # GAP → Dense(FC) → ReLU → Dropout → Output
    layers.GlobalAveragePooling2D(),
    layers.Dense(128),
    layers.ReLU(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [ ]:
model.summary()
parameter_count = model.count_params()
print("Total parameters:", f"{parameter_count:,}")

## [5] Optimizer 설정

- **Adam**
- Learning Rate: **0.001**

In [ ]:
# [5] Optimizer 설정
optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

BEST_MODEL_PATH = OUTPUT_DIR / "best_val_loss.keras"
checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=BEST_MODEL_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

## [6] 학습

**EarlyStopping을 사용하지 않고 30 Epoch 전체를 학습**합니다.  
대신 Validation Loss가 가장 낮았던 모델을 별도로 저장해  
`Final Epoch vs Best Validation Loss`를 나중에 비교할 수 있게 합니다.

In [ ]:
# [6] 학습
# 모든 Furniture 실험은 비교를 위해 30 Epoch를 끝까지 수행합니다.
start_time = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint_cb],
    verbose=1
)
training_seconds = time.time() - start_time

## [7] 성능 평가

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.index = np.arange(1, len(history_df) + 1)
history_df.index.name = "epoch"

best_epoch = int(history_df["val_loss"].idxmin())
best_val_loss = float(history_df.loc[best_epoch, "val_loss"])
best_val_accuracy = float(history_df["val_accuracy"].max())

def dataset_labels(ds):
    labels = []
    for _, y in ds:
        labels.extend(y.numpy())
    return np.array(labels)

y_test = dataset_labels(test_ds)

final_probs = model.predict(test_ds, verbose=0)
final_pred = np.argmax(final_probs, axis=1)

best_model = keras.models.load_model(BEST_MODEL_PATH)
best_probs = best_model.predict(test_ds, verbose=0)
best_pred = np.argmax(best_probs, axis=1)

def evaluate_predictions(y_true, y_pred, prefix):
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    report = pd.DataFrame(
        classification_report(
            y_true, y_pred, target_names=CLASS_NAMES,
            output_dict=True, zero_division=0
        )
    ).T
    report.to_csv(OUTPUT_DIR / f"{prefix}_classification_report.csv", encoding="utf-8-sig")

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
        ax=ax, xticks_rotation=45, values_format="d"
    )
    ax.set_title(f"{EXPERIMENT_NAME} - {prefix} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_confusion_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()

    return {
        f"{prefix}_test_accuracy": acc,
        f"{prefix}_macro_precision": p,
        f"{prefix}_macro_recall": r,
        f"{prefix}_macro_f1": f1
    }

final_metrics = evaluate_predictions(y_test, final_pred, "final")
best_metrics = evaluate_predictions(y_test, best_pred, "best")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["accuracy"], label="Train Accuracy")
plt.plot(history_df.index, history_df["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"{EXPERIMENT_NAME} - Accuracy")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_curve.png", dpi=160, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(history_df.index, history_df["loss"], label="Train Loss")
plt.plot(history_df.index, history_df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()

## [8] 결과 저장

In [ ]:
history_df.to_csv(OUTPUT_DIR / "history.csv", encoding="utf-8-sig")

summary = {
    "experiment": EXPERIMENT_NAME,
    "dataset": "Furniture",
    "framework": "Keras",
    "optimizer": "Adam",
    "model_version": "Dropout",
    "seed": SEED,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM if "Adam" == "SGD" else np.nan,
    "dropout": 0.5,
    "spatial_dropout_1": np.nan,
    "spatial_dropout_2": np.nan,
    "fc_units": 128,
    "parameter_count": parameter_count,
    "training_seconds": training_seconds,
    "best_epoch_by_val_loss": best_epoch,
    "best_val_loss": best_val_loss,
    "best_val_accuracy": best_val_accuracy,
    "final_train_accuracy": float(history_df.iloc[-1]["accuracy"]),
    "final_val_accuracy": float(history_df.iloc[-1]["val_accuracy"]),
    "final_train_val_gap": float(history_df.iloc[-1]["accuracy"] - history_df.iloc[-1]["val_accuracy"]),
    "model_definition_code_lines": 21,
    "training_code_lines": 9,
    **final_metrics,
    **best_metrics
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False, encoding="utf-8-sig")
display(summary_df.T)

zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Saved:", zip_path)

## [9] 결과 ZIP 자동 다운로드

Google Colab에서 실행 중이라면, 저장된 실험 결과 ZIP을 브라우저로 자동 다운로드합니다.

In [ ]:
# ------------------------------------------------------------
# [9] Colab에서 결과 ZIP 자동 다운로드
# 학습/평가/저장이 끝난 뒤 생성된 ZIP 파일을 브라우저로 다운로드합니다.
# ------------------------------------------------------------
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Google Colab 환경이 아니므로 자동 다운로드를 실행하지 않았습니다.")
    print("결과 ZIP 경로:", zip_path)